
## Expected inputs

### 1) Port dictionary

This version is adapted to dictionaries shaped like your sample:

```python
ports_dict = {
    "SEVST": {
        "port_importance_metric": 1,
        "country_code": "SE",
        "port_metadata": {
            "name": "VASTERAS",
            "unlocode": "SEVST",
            "Harbor Size": "Medium",
            "Harbor Type": "Canal or Lake",
            "Facilities - Container": "Yes",
            ...
        },
        "final_data_wo_details": [
            {
                "code": "465e5f61",
                "type": "anchorage",
                "polygon": [(59.5149, 16.7136)],   # point-like (lat, lon) in your sample
                "centroid": [(59.5149, 16.7136)],
                ...
            },
            ...
        ],
        "osm": [
            {
                "facility": {
                    "geographicalFeature": {
                        "type": "FeatureCollection",
                        "features": [... polygon features ...],
                    }
                }
            },
            ...
        ],
        "port_boundary_polygon": <shapely Polygon>,
        "convex_hull_centre": {"lat": 59.5940, "lon": 16.56},
        "convex_hull_radius_nm": 5.26,
    }
}
```

In [2]:

from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path
from typing import Any

import geopandas as gpd
from IPython.display import display
import joblib
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely import wkt
from shapely.geometry import GeometryCollection, MultiPolygon, Point, Polygon, mapping, shape
from shapely.geometry.base import BaseGeometry
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Geometry is in a geographic CRS")
pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True, precision=4)


c:\Users\Niru\Downloads\Projects\TauML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

# -----------------------------
# User configuration
# -----------------------------
RANDOM_STATE = 42
EXPECTED_EMBED_DIM = 64
YEARS = [2024]                 # keep as a list even for one year

# Port dictionary input
PORTS_DICT_PATH = "../data/sample_port_dict.pkl"   # change to your own .pkl or .json path
PORT_COORD_ORDER = "latlon"       # your sample point tuples look like (lat, lon)

# Which geometry should define a "part"?
# - "port_boundary": safest default for this schema; one part per port boundary polygon
# - "osm_facility_features": use OSM facility polygons nested under payload["osm"]
# - "metadata_polygons": try port_metadata["port_polygon_smdg"] / ["port_polygon_bic"]
# - "final_data_polygons": use polygons from final_data_wo_details (or point buffers if enabled)
# - "auto": explicit parts -> port boundary -> metadata polygons -> OSM polygons -> final_data polygons
PART_GEOMETRY_MODE = "port_boundary"
POINT_BUFFER_RADIUS_M = None       # e.g. 150.0 to turn point-like final_data records into small polygons
MAX_PARTS_PER_PORT = None          # e.g. 256 if you use many OSM facility polygons and want to cap runtime

# AlphaEarth / GCS input
AEF_BUCKET_ROOT =  "gs://alphaearth_annual_embed_2025/satellite_embedding/v1/annual/"
# AEF_INDEX_URI = f"{AEF_BUCKET_ROOT}/aef_index.parquet" 
AEF_INDEX_URI = "gs://alphaearth_foundations/satellite_embedding/v1/annual/aef_index.parquet"
TILE_URI_COL = "tile_uri"     # change only if your index uses a different name

# If you use Google's official public bucket, set your billing project here.
# The official bucket is requester-pays.
GCS_USER_PROJECT = 'tau-agentic-priors'       # e.g. "my-gcp-project"
# GOOGLE_APPLICATION_CREDENTIALS = '/tmp/tmp.U8n76087Z7/application_default_credentials.json'  # e.g. "/path/to/service-account.json"
GOOGLE_APPLICATION_CREDENTIALS = '/tmp/tmp.gL0jBqFjPd/application_default_credentials.json'
GCS_TOKEN = "cloud"           # default: use Application Default Credentials

# TIFF decoding mode:
# - "official_int8_cog": raw official AlphaEarth COGs, signed int8, needs de-quantization
# - "float_embeddings": your TIFFs are already float embeddings in [-1, 1]
TIFF_VALUE_MODE = "official_int8_cog"

# Aggregation / sampling
ALL_TOUCHED = False
MAX_PATCHES_PER_PART = 12000          # cap after extraction to keep memory bounded
PROTOTYPE_SAMPLES_PER_PART = 1500     # used only to fit global prototype centers
MAX_PROTOTYPE_SAMPLES_TOTAL = 250000
N_PROTOTYPES = 32
PORT_WEIGHT_COL = "n_total_valid_pixels"   # or "part_area_m2"
PCA_TARGET_DIM = 128                  # used only if enough ports are available

# Metadata targets to train heads on.
# Fill these after inspecting `port_metadata_df.columns`.
TARGET_COLS = []                      # e.g. ["Harbor Size", "Harbor Type", "Facilities - Container"]
CV_GROUP_COL = "port_id"             # recommended when multiple years are present

OUTPUT_DIR = Path("outputs/alphaearth_port_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(RANDOM_STATE)



## Load your port dictionary

You can either:

- point `PORTS_DICT_PATH` at a JSON or pickle file, or
- assign `ports_dict` directly in the next cell.

Pickle is the easiest option when the dictionary already contains shapely objects such as `port_boundary_polygon`.


In [23]:

ports_dict: dict[str, Any] | None = None

if PORTS_DICT_PATH is not None:
    path = Path(PORTS_DICT_PATH)
    if path.suffix.lower() == ".json":
        ports_dict = json.loads(path.read_text())
    elif path.suffix.lower() in {".pkl", ".pickle"}:
        with open(path, "rb") as f:
            ports_dict = pickle.load(f)
    else:
        raise ValueError(f"Unsupported port dictionary format: {path.suffix}")

# Or paste / construct the dictionary here directly.
# ports_dict = {...}

if ports_dict is None:
    raise ValueError(
        "Load your port dictionary by setting `PORTS_DICT_PATH` or assigning `ports_dict` directly in this cell."
    )

print(f"Loaded {len(ports_dict):,} ports from the dictionary.")


Loaded 100 ports from the dictionary.


In [24]:

def _flatten_dict(d: dict[str, Any] | None, sep: str = "__") -> dict[str, Any]:
    if not d:
        return {}
    return pd.json_normalize(d, sep=sep).to_dict(orient="records")[0]


def _is_scalar_like(v: Any) -> bool:
    scalar_types = (str, int, float, bool, np.integer, np.floating, np.bool_)
    return v is None or isinstance(v, scalar_types)


def _point_pair_to_lonlat(pt: Any, coord_order: str = "lonlat") -> tuple[float, float]:
    if isinstance(pt, dict):
        lon_keys = ["lon", "longitude", "x"]
        lat_keys = ["lat", "latitude", "y"]
        lon = next((pt[k] for k in lon_keys if k in pt), None)
        lat = next((pt[k] for k in lat_keys if k in pt), None)
        if lon is None or lat is None:
            raise ValueError(f"Could not parse lon/lat from point dict: {pt}")
        return float(lon), float(lat)

    if not isinstance(pt, (list, tuple, np.ndarray)) or len(pt) != 2:
        raise ValueError(f"Expected a 2-element coordinate pair, got {pt!r}")

    a, b = float(pt[0]), float(pt[1])
    if coord_order == "lonlat":
        return a, b
    if coord_order == "latlon":
        return b, a
    raise ValueError("coord_order must be either 'lonlat' or 'latlon'")


def _coords_look_like_ring(coords: Any) -> bool:
    if not isinstance(coords, (list, tuple, np.ndarray)) or len(coords) == 0:
        return False
    first = coords[0]
    if isinstance(first, dict):
        return True
    return isinstance(first, (list, tuple, np.ndarray)) and len(first) == 2 and not isinstance(
        first[0], (list, tuple, dict, np.ndarray)
    )


def _coords_look_like_polygon_coords(coords: Any) -> bool:
    if not isinstance(coords, (list, tuple, np.ndarray)) or len(coords) == 0:
        return False
    first = coords[0]
    return _coords_look_like_ring(first)


def _coerce_polygon(coords: Any, coord_order: str = "lonlat") -> Polygon:
    if _coords_look_like_ring(coords):
        exterior = [_point_pair_to_lonlat(pt, coord_order=coord_order) for pt in coords]
        if len(exterior) < 3:
            raise ValueError("A polygon ring needs at least 3 coordinates.")
        poly = Polygon(exterior)
    elif _coords_look_like_polygon_coords(coords):
        rings = []
        for ring in coords:
            ring_lonlat = [_point_pair_to_lonlat(pt, coord_order=coord_order) for pt in ring]
            if len(ring_lonlat) < 3:
                raise ValueError("A polygon ring needs at least 3 coordinates.")
            rings.append(ring_lonlat)
        poly = Polygon(rings[0], holes=rings[1:] or None)
    else:
        raise ValueError(f"Could not parse polygon coordinates from object of type {type(coords)}")

    if not poly.is_valid:
        poly = poly.buffer(0)
    if poly.is_empty:
        raise ValueError("Polygon became empty after validity repair")
    return poly


def _coerce_geometry(obj: Any, coord_order: str = "lonlat") -> BaseGeometry | None:
    if obj is None:
        return None

    if isinstance(obj, BaseGeometry):
        geom = obj
    elif isinstance(obj, str):
        try:
            geom = wkt.loads(obj)
        except Exception:
            return None
    elif isinstance(obj, dict) and "type" in obj:
        try:
            geom = shape(obj)
        except Exception:
            return None
    elif isinstance(obj, (list, tuple, np.ndarray)):
        try:
            geom = _coerce_polygon(obj, coord_order=coord_order)
        except Exception:
            return None
    else:
        return None

    if geom is None or geom.is_empty:
        return None
    if not geom.is_valid:
        geom = geom.buffer(0)
    return geom if not geom.is_empty else None


def _iter_polygons(geom: BaseGeometry | None):
    if geom is None or geom.is_empty:
        return
    if isinstance(geom, Polygon):
        poly = geom if geom.is_valid else geom.buffer(0)
        if not poly.is_empty:
            yield poly
    elif isinstance(geom, MultiPolygon):
        for sub_geom in geom.geoms:
            yield from _iter_polygons(sub_geom)
    elif isinstance(geom, GeometryCollection):
        for sub_geom in geom.geoms:
            yield from _iter_polygons(sub_geom)


def _buffer_lonlat_point(lon: float, lat: float, radius_m: float) -> Polygon:
    gs = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326")
    return gs.to_crs("EPSG:6933").buffer(radius_m).to_crs("EPSG:4326").iloc[0]


def _extract_explicit_parts(port_id: str, payload: dict[str, Any], coord_order: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    parts = payload.get("parts") or payload.get("hulls") or payload.get("areas") or []

    for i, part in enumerate(parts):
        coords = part.get("polygon_lonlat") or part.get("coords") or part.get("hull") or part.get("polygon")
        geom = _coerce_geometry(coords, coord_order=coord_order)
        if geom is None:
            continue

        for k, poly in enumerate(_iter_polygons(geom)):
            row = {
                "port_id": port_id,
                "part_id": part.get("part_id", f"{port_id}__part_{i:03d}_{k:02d}"),
                "geometry": poly,
                "part_source": "explicit_parts",
            }
            for key, value in part.items():
                if key not in {"polygon_lonlat", "coords", "hull", "polygon"}:
                    row[key] = value
            rows.append(row)
    return rows


def _extract_port_boundary(port_id: str, payload: dict[str, Any], coord_order: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for key in ["port_boundary_polygon", "port_boundary_polygon_original"]:
        geom = _coerce_geometry(payload.get(key), coord_order=coord_order)
        if geom is None:
            continue
        for i, poly in enumerate(_iter_polygons(geom)):
            rows.append(
                {
                    "port_id": port_id,
                    "part_id": f"{port_id}__boundary_{i:03d}",
                    "geometry": poly,
                    "part_source": key,
                }
            )
        if rows:
            return rows
    return rows


def _extract_metadata_polygons(port_id: str, payload: dict[str, Any], coord_order: str) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    metadata = payload.get("port_metadata") or payload.get("metadata") or {}

    for key in ["port_polygon_smdg", "port_polygon_bic"]:
        geom = _coerce_geometry(metadata.get(key), coord_order=coord_order)
        if geom is None:
            continue
        for i, poly in enumerate(_iter_polygons(geom)):
            rows.append(
                {
                    "port_id": port_id,
                    "part_id": f"{port_id}__{key}_{i:03d}",
                    "geometry": poly,
                    "part_source": key,
                }
            )
    return rows


def _extract_osm_facility_features(port_id: str, payload: dict[str, Any]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    osm_items = payload.get("osm") or []

    for i, item in enumerate(osm_items):
        facility = (item or {}).get("facility") or {}
        geographical_feature = facility.get("geographicalFeature") or {}
        if not isinstance(geographical_feature, dict) or geographical_feature.get("type") != "FeatureCollection":
            continue

        features = geographical_feature.get("features") or []
        for j, feature in enumerate(features):
            geom = _coerce_geometry((feature or {}).get("geometry"), coord_order="lonlat")
            if geom is None:
                continue

            props = (feature or {}).get("properties") or {}
            feature_metadata = {
                "facility_name": facility.get("name"),
                "code": item.get("code"),
                "codeProvider": item.get("codeProvider"),
                "facility_category": props.get("category"),
                "operator__name": ((item.get("operator") or {}).get("name")),
            }
            osm_tags = props.get("osm_tags")
            if isinstance(osm_tags, dict):
                for tag_key, tag_value in osm_tags.items():
                    feature_metadata[f"osm_tag__{tag_key}"] = tag_value

            for k, poly in enumerate(_iter_polygons(geom)):
                row = {
                    "port_id": port_id,
                    "part_id": f"{port_id}__osm_{i:04d}_{j:02d}_{k:02d}",
                    "geometry": poly,
                    "part_source": "osm_facility_features",
                }
                row.update(feature_metadata)
                rows.append(row)
    return rows


def _extract_final_data_polygons(
    port_id: str,
    payload: dict[str, Any],
    coord_order: str,
    point_buffer_radius_m: float | None,
) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    items = payload.get("final_data_wo_details") or []

    for i, item in enumerate(items):
        geom = _coerce_geometry((item or {}).get("polygon"), coord_order=coord_order)
        source_name = "final_data_polygons"

        if geom is None and point_buffer_radius_m is not None:
            point_like = (item or {}).get("polygon") or (item or {}).get("centroid")
            if isinstance(point_like, (list, tuple)) and len(point_like) == 1:
                try:
                    lon, lat = _point_pair_to_lonlat(point_like[0], coord_order=coord_order)
                    geom = _buffer_lonlat_point(lon, lat, radius_m=point_buffer_radius_m)
                    source_name = "final_data_point_buffer"
                except Exception:
                    geom = None

        if geom is None:
            continue

        for k, poly in enumerate(_iter_polygons(geom)):
            row = {
                "port_id": port_id,
                "part_id": f"{port_id}__final_{i:03d}_{k:02d}",
                "geometry": poly,
                "part_source": source_name,
            }
            for key, value in item.items():
                if key not in {"polygon", "centroid"}:
                    row[key] = value
            rows.append(row)
    return rows


def _extract_parts_for_port(
    port_id: str,
    payload: dict[str, Any],
    coord_order: str,
    geometry_mode: str,
    point_buffer_radius_m: float | None,
) -> list[dict[str, Any]]:
    extractors = {
        "explicit_parts": lambda: _extract_explicit_parts(port_id, payload, coord_order),
        "port_boundary": lambda: _extract_port_boundary(port_id, payload, coord_order),
        "metadata_polygons": lambda: _extract_metadata_polygons(port_id, payload, coord_order),
        "osm_facility_features": lambda: _extract_osm_facility_features(port_id, payload),
        "final_data_polygons": lambda: _extract_final_data_polygons(
            port_id, payload, coord_order, point_buffer_radius_m
        ),
    }

    if geometry_mode == "auto":
        priority = [
            "explicit_parts",
            "port_boundary",
            "metadata_polygons",
            "osm_facility_features",
            "final_data_polygons",
        ]
        for name in priority:
            rows = extractors[name]()
            if rows:
                return rows
        return []

    if geometry_mode not in extractors:
        raise ValueError(f"Unsupported PART_GEOMETRY_MODE: {geometry_mode}")

    rows = extractors[geometry_mode]()
    if rows:
        return rows

    if geometry_mode != "port_boundary":
        fallback_rows = _extract_port_boundary(port_id, payload, coord_order)
        if fallback_rows:
            warnings.warn(
                f"Port {port_id}: no parts found for mode {geometry_mode!r}; falling back to port_boundary_polygon."
            )
            return fallback_rows

    return rows


def ports_dict_to_frames(
    ports_dict: dict[str, Any],
    coord_order: str = "lonlat",
    crs: str = "EPSG:4326",
    geometry_mode: str = "auto",
    point_buffer_radius_m: float | None = None,
    max_parts_per_port: int | None = None,
) -> tuple[gpd.GeoDataFrame, pd.DataFrame]:
    """Turn the nested port dictionary into:

    1) a GeoDataFrame with one row per port part
    2) a DataFrame with one row per port metadata payload
    """
    part_rows: list[dict[str, Any]] = []
    metadata_rows: list[dict[str, Any]] = []

    complex_keys = {
        "parts",
        "hulls",
        "areas",
        "final_data_wo_details",
        "smdg",
        "osm",
        "bic",
        "clustering_algo",
        "gfw_data",
        "port_boundary_polygon",
        "port_boundary_polygon_original",
        "port_metadata",
        "metadata",
    }

    for port_id, payload in ports_dict.items():
        payload = payload or {}

        metadata = {"port_id": port_id}
        metadata.update(_flatten_dict(payload.get("port_metadata") or payload.get("metadata") or {}))

        if isinstance(payload.get("convex_hull_centre"), dict):
            for key, value in payload["convex_hull_centre"].items():
                metadata[f"convex_hull_centre__{key}"] = value

        for key, value in payload.items():
            if key in complex_keys:
                continue
            if _is_scalar_like(value) and key not in metadata:
                metadata[key] = value
        metadata_rows.append(metadata)

        port_part_rows = _extract_parts_for_port(
            port_id=port_id,
            payload=payload,
            coord_order=coord_order,
            geometry_mode=geometry_mode,
            point_buffer_radius_m=point_buffer_radius_m,
        )
        part_rows.extend(port_part_rows)

    parts_gdf = gpd.GeoDataFrame(part_rows, geometry="geometry", crs=crs)
    if parts_gdf.empty:
        raise ValueError("No valid parts were found in the port dictionary.")

    parts_gdf["part_area_m2"] = parts_gdf.to_crs("EPSG:6933").area.astype(float)

    if max_parts_per_port is not None:
        parts_gdf = (
            parts_gdf.sort_values(["port_id", "part_area_m2"], ascending=[True, False])
            .groupby("port_id", group_keys=False)
            .head(max_parts_per_port)
            .copy()
        )

    metadata_df = pd.DataFrame(metadata_rows).drop_duplicates(subset=["port_id"])
    return parts_gdf, metadata_df


In [25]:

parts_gdf, port_metadata_df = ports_dict_to_frames(
    ports_dict,
    coord_order=PORT_COORD_ORDER,
    geometry_mode=PART_GEOMETRY_MODE,
    point_buffer_radius_m=POINT_BUFFER_RADIUS_M,
    max_parts_per_port=MAX_PARTS_PER_PORT,
)

print(f"Parts: {len(parts_gdf):,}")
print(f"Ports with metadata rows: {len(port_metadata_df):,}")
if "part_source" in parts_gdf.columns:
    display(parts_gdf["part_source"].value_counts(dropna=False).rename_axis("part_source").to_frame("n_parts"))

display(parts_gdf.head())
display(port_metadata_df.head())


Parts: 99
Ports with metadata rows: 100


,n_parts
part_source,
port_boundary_polygon,99


,port_id,part_id,geometry,part_source,part_area_m2
0,SEVST,SEVST__boundary_000,"POLYGON ((16.56 59.67718, 16.57615 59.67678, 1...",port_boundary_polygon,2.698347e+08
1,NLIJM,NLIJM__boundary_000,"POLYGON ((4.78778 52.36736, 4.78501 52.36737, ...",port_boundary_polygon,6.018445e+08
2,CNMWN,CNMWN__boundary_000,"POLYGON ((114.10337 21.98889, 114.10039 21.988...",port_boundary_polygon,5.048687e+09
3,IDMKQ,IDMKQ__boundary_000,"POLYGON ((140.3728 -8.38439, 140.38104 -8.3847...",port_boundary_polygon,2.672317e+08
4,SLFNA,SLFNA__boundary_000,"POLYGON ((-13.44069 8.32626, -13.44385 8.32647...",port_boundary_polygon,1.163352e+09


,port_id,uuid,name,unlocode,port_type,country,lat,lon,area_lvl1,area_lvl2,Harbor Size,Harbor Type,Shelter Afforded,Channel Depth (m),Anchorage Depth (m),Cargo Pier Depth (m),Oil Terminal Depth (m),Liquified Natural Gas Terminal Depth (m),Tidal Range (m),Maximum Vessel Draft (m),Entrance Restriction - Tide,Entrance Restriction - Heavy Swell,Pilotage - Compulsory,Tugs - Assistance,Vessel Traffic Service,Good Holding Ground,Turning Area,Facilities - Container,Cranes - Mobile,Cranes Container,Repairs,Dry Dock,Railway,port_features_text,port_polygon_smdg,port_polygon_bic,convex_hull_centre__lat,convex_hull_centre__lon,port_importance_metric,country_code,total_smdg_berths_count,total_osm_berths_count,total_osm_anchorage_count,total_clustering_berths_count,total_clustering_anchorage_count,total_clustering_congestion_count,total_gfw_berth_count,total_gfw_anchorage_count,convex_hull_radius_nm
0,SEVST,fa50e870-2abe-256a-f3ff-604bdf05d2ff,VASTERAS,SEVST,Port,SE,59.594000,16.560000,Baltic Sea,Baltic Sea,Medium,Canal or Lake,Good,4.9,6.4,3.4,4.9,0.0,0.0,7.0,No,No,Yes,Yes,Unknown,Unknown,Unknown,Yes,Yes,Unknown,Moderate,Unknown,Small,A canal or lake port. of Medium size. offering...,[],[],59.594025,16.560000,1,SE,0,0,0,0,0,0,0,19,5.263218
1,NLIJM,b4f5871b-e943-d10e-2fb5-dee06e8ce998,IJMUIDEN,NLIJM,Port,NL,52.461420,4.583818,UK Coast & Atlantic,North Sea,Small,Coastal (Tide Gates),Good,14.0,14.0,9.4,0.0,0.0,2.0,0.0,No,No,Yes,Yes,Unknown,Unknown,Yes,Unknown,Yes,Unknown,Limited,Unknown,Small,A coastal (tide gates) port. of Small size. of...,[],"[[(52.461213193302505, 4.581593420530442), (52...",52.457122,4.511280,2,NL,0,0,2,0,0,6,0,41,14.787890
2,CNMWN,219c0002-9227-4dfa-87d3-cba3daa45c8c,Mawan Pt,CNMWN,Port,CN,22.450000,113.883330,South China,South China,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Port characteristics not available.,[],[],22.378156,113.986655,12,NaN,0,11,62,0,0,0,0,371,33.027270
3,IDMKQ,8a10a0a1-bf84-a002-3aab-6cbf6427c34e,MERAUKE,IDMKQ,Port,ID,-8.467576,140.372800,Indonesia,Arafura Sea,Very Small,River (Natural),Fair,4.9,6.4,3.4,4.9,0.0,3.0,6.0,No,No,Yes,Yes,Unknown,Yes,Unknown,Unknown,Yes,Unknown,Limited,Unknown,Unknown,A river (natural) port. of Very Small size. of...,[],[],-8.467578,140.372800,5,ID,0,0,0,0,0,0,0,5,5.250412
4,SLFNA,7256c14d-cee1-0eb1-5453-fa5f871ce312,FREETOWN,SLFNA,Port,SL,8.492001,-13.223000,West Africa,West Africa,Small,Coastal (Natural),Fair,7.9,14.0,9.4,11.0,0.0,3.0,0.0,Yes,Yes,Yes,Yes,Unknown,Yes,Unknown,Unknown,Yes,Unknown,Moderate,Unknown,Small,A coastal (natural) port. of Small size. offer...,[],[],8.464058,-13.454203,110,SL,0,0,0,2,0,3,0,21,18.856411



## Load the AlphaEarth tile index


In [29]:

def _storage_options(project: str | None = None, token: str | None = "cloud") -> dict[str, Any] | None:
    opts: dict[str, Any] = {}
    if token is not None:
        opts["token"] = token
    if project is not None:
        opts["project"] = project
    return opts or None


def load_alphaearth_index(
    index_uri: str,
    project: str | None = None,
    token: str | None = "cloud",
) -> gpd.GeoDataFrame:
    """Load the published AEF index or a compatible custom index."""
    storage_options = _storage_options(project=project, token=token)

    if index_uri.endswith(".parquet"):
        gdf = gpd.read_parquet(index_uri, storage_options=storage_options)
    elif index_uri.endswith(".csv"):
        df = pd.read_csv(index_uri, storage_options=storage_options)
        if "geometry" in df.columns:
            geometry = gpd.GeoSeries.from_wkt(df["geometry"], crs="EPSG:4326")
            df = df.drop(columns=["geometry"])
        elif "WKT" in df.columns:
            geometry = gpd.GeoSeries.from_wkt(df["WKT"], crs="EPSG:4326")
            df = df.drop(columns=["WKT"])
        else:
            raise ValueError("CSV index needs either a `geometry` WKT column or the official `WKT` column.")
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    else:
        raise ValueError("AEF_INDEX_URI should point to a .parquet or .csv file.")

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf


def ensure_tile_uri_column(
    index_gdf: gpd.GeoDataFrame,
    bucket_root: str,
    tile_uri_col: str = "tile_uri",
) -> gpd.GeoDataFrame:
    """Try to create `tile_uri` from common index schemas.

    If this helper cannot infer the path from your index, create the column manually
    in the cell below and re-run.
    """
    out = index_gdf.copy()
    if tile_uri_col in out.columns:
        return out

    common_absolute = ["uri", "gcs_uri", "file_uri"]
    common_relative = ["path", "relative_path", "blob_name", "filename", "file", "basename", "name"]

    for col in common_absolute:
        if col in out.columns:
            out[tile_uri_col] = out[col].astype(str)
            return out

    for col in common_relative:
        if col in out.columns:
            s = out[col].astype(str)
            if s.str.startswith("gs://").all():
                out[tile_uri_col] = s
                return out
            if {"year", "utm_zone"}.issubset(out.columns):
                out[tile_uri_col] = (
                    bucket_root.rstrip("/")
                    + "/"
                    + out["year"].astype(str)
                    + "/"
                    + out["utm_zone"].astype(str)
                    + "/"
                    + s.str.lstrip("/")
                )
                return out
            out[tile_uri_col] = bucket_root.rstrip("/") + "/" + s.str.lstrip("/")
            return out

    raise ValueError(
        "Could not infer a tile URI column from the index. Add one manually, named `tile_uri`, before continuing."
    )

index_gdf = load_alphaearth_index(AEF_INDEX_URI, project=GCS_USER_PROJECT, token=GCS_TOKEN)
index_gdf = ensure_tile_uri_column(index_gdf, bucket_root=AEF_BUCKET_ROOT, tile_uri_col=TILE_URI_COL)

required_index_cols = {"year", TILE_URI_COL, "geometry"}
missing_index_cols = required_index_cols - set(index_gdf.columns)
if missing_index_cols:
    raise ValueError(f"The AlphaEarth index is missing required columns: {missing_index_cols}")

index_gdf = index_gdf[index_gdf["year"].isin(YEARS)].copy()
index_gdf["tile_uri"] = index_gdf[TILE_URI_COL].astype(str)

print(f"Index rows in selected years: {len(index_gdf):,}")
display(index_gdf[[c for c in ["year", "utm_zone", TILE_URI_COL] if c in index_gdf.columns]].head())


ValueError: Invalid gcloud credentials